# Варіант: [Впишіть ваше місто / назву товару]
Практичне заняття: Очищення брудного набору даних (Data Cleaning)

In [ ]:
import pandas as pd
import numpy as np

# Побудова брудного DataFrame з 8 рядків та 5 дефектами
raw_data = [
    {"id": 1, "місто": "Київ", "ціна": "800 грн", "кількість": 2},            # Дефект 3: " грн"
    {"id": 2, "місто": " Київ ", "ціна": 800, "кількість": np.nan},            # Дефект 1: NaN, Дефект 4: пробіли
    {"id": 3, "місто": "КИЇВ", "ціна": "850 грн", "кількість": 3},            # Дефект 3: " грн", Дефект 4: ВЕЛИКІ ЛІТЕРИ
    {"id": 3, "місто": "КИЇВ", "ціна": "850 грн", "кількість": 3},            # Дефект 2: Повний дублікат рядка id=3
    {"id": 4, "місто": "Київ", "ціна": 800, "кількість": np.nan},            # Дефект 1: NaN
    {"id": 5, "місто": "Київ", "ціна": 7200, "кількість": 10},               # Дефект 5: Викид (ціна х9 від базової)
    {"id": 6, "місто": "Київ", "ціна": 800, "кількість": 1},
    {"id": 7, "місто": "Київ", "ціна": 800, "кількість": 4}
]

df = pd.DataFrame(raw_data)

print("--- Початковий брудний DataFrame (8 рядків) ---")
display(df)

### Завдання 2. Обробка пропусків (NaN)

In [ ]:
# 1. Визначення кількості пропусків у кожному стовпці
print("--- Кількість пропусків по стовпцях ---")
print(df.isna().sum())

# 2. Обчислення медіани та заповнення пропусків
median_qty = df["кількість"].median()
print(f"\nМедіана для стовпця 'кількість': {median_qty}")

df["кількість"].fillna(median_qty, inplace=True)

print("\n--- Після заповнення пропусків у 'кількість' ---")
display(df)

**Обґрунтування вибору медіани замість середнього:**
У малій вибірці (всього 8 спостережень) середнє значення надзвичайно чутливе до можливих викидів у кількості товару (наприклад, гуртове замовлення на 10 одиниць). Медіана є стійкою (робастною) метрикою центральної тенденції, тому вона краще відображає типове замовлення пересічного покупця і не спотворює розподіл.

### Завдання 3. Виявлення та видалення дублікатів

In [ ]:
# 1. Пошук дублікатів без subset (повний збіг усіх полів)
full_duplicates = df.duplicated().sum()
print(f"Повних дублікатів рядків (без subset): {full_duplicates}")

# 2. Пошук дублікатів з subset за унікальним ключем id
id_duplicates = df.duplicated(subset=["id"]).sum()
print(f"Дублікатів за ID (subset=['id']): {id_duplicates}")

# 3. Видалення дублікатів
df.drop_duplicates(subset=["id"], inplace=True)

print(f"\nКількість рядків після видалення дублікатів: {len(df)}")
display(df)

### Завдання 4. Приведення типів даних та узгодження категорій

In [ ]:
# 1. Очищення стовпця "ціна" та конвертація у float
df["ціна"] = df["ціна"].astype(str).str.replace(" грн", "").astype(float)

# 2. Нормалізація назви міста: прибираємо пробіли та зводимо до канонічного вигляду
df["місто"] = df["місто"].str.strip().str.capitalize()

print("--- Типи даних після приводу ---")
print(df.dtypes)

print("\nУнікальні значення міста:", df["місто"].unique())

print("\n--- Очищена таблиця ---")
display(df)

### Завдання 5. Аналіз викидів (IQR)

In [ ]:
# Обчислення квантилів та IQR для очищеного стовпця "ціна"
q1 = df["ціна"].quantile(0.25)
q3 = df["ціна"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
print(f"Нижня межа IQR: {lower_bound}, Верхня межа IQR: {upper_bound}")

# Знаходження рядків, які виходять за межі IQR
outliers = df[(df["ціна"] < lower_bound) | (df["ціна"] > upper_bound)]
print("\n--- Виявлені викиди за правилом IQR ---")
display(outliers)

**Рішення щодо аномального значення ціни (7200 грн):**
Вибране рішення: **Вважати помилкою введення (оператора) та відкоригувати/видалити.**

*Обґрунтування:* Оскільки мова йде про один і той самий товар ("Кавомашина"), базова ціна якого становить близько 800 грн, значення 7200 грн найімовірніше є наслідком технічної помилки (наприклад, випадково приписали зайвий нуль або вказали ціну за комплект із 9 штуками в полі одиничної ціни). Якщо це одинична роздрібна позиція товару, заміна або вилучення цього запису запобіжить викривленню середньої чекової вартості позиції.

### Відповіді на підсумкові питання:

**1. Чи можете пояснити, чому заповнення пропуску середнім (а не медіаною) може згодом спотворити стандартне відхилення?**
Заповнення пропусків середнім значенням штучно концентрує дані навколо центру розподілу. Оскільки кожне імпутоване значення дорівнює самому середньому, його відхилення від середнього дорівнює нулю. Це штучно **занижує** дисперсію та стандартне відхилення. Крім того, якщо в оригінальних даних є аномальні викиди, саме середнє виявиться зміщеним у бік викиду, і всі заповнені значення затягнуть аналіз у хибну сторону.

**2. Чи розумієте різницю між duplicated() без subset і з subset — чому результат може відрізнятись?**
- `duplicated()` **без `subset`** перевіряє повний збіг усього рядка по всіх стовпцях одночасно. Рядок вважається дублікатом лише тоді, коли абсолютно кожне значення збігається.
- `duplicated(subset=[...])` перевіряє унікальність лише за вказаними ключовими стовпцями (наприклад, `id`). Якщо в базі є два рядки з однаковим `id`, але різними датами або коментарями, варіант без `subset` їх не помітить, а з `subset=['id']` — коректно визначить як дублікат за бізнес-логікою.

**3. Чи можете пояснити, чому автоматичне видалення всіх значень поза межами IQR не завжди правильне рішення?**
Автоматичне видалення за IQR не враховує контекст даних. Значення поза межами `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` не обов'язково є помилками — це можуть бути реальні рідкісні події (наприклад, оптова покупка B2B, сезонний сплеск продажів у Чорну п'ятницю тощо). Автоматичне видалення таких даних призведе до втрати цінного бізнес-сигналу та штучного приховування справжньої мінливості процесів.